# Fix Backend URL Placeholder and Redeploy

This notebook fixes the BACKEND_URL_PLACEHOLDER issue in the deployed app.

**Issue**: The JavaScript bundle contains `BACKEND_URL_PLACEHOLDER` instead of the actual app URL, causing API calls to fail.

**Solution**: Replace the placeholder and redeploy the app.

In [ ]:
%run ./0-Parameters

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.apps import AppDeployment, AppDeploymentArtifacts, AppDeploymentMode
import uuid
import os

w = WorkspaceClient()

In [ ]:
# Get the app URL
app = w.apps.get(APP_NAME)
print(f"App Name: {app.name}")
print(f"App URL: {app.url}")
print(f"App Status: {app.status}")

In [ ]:
# Determine paths
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
parent_path = os.path.dirname(notebook_path)
grandparent_path = os.path.dirname(parent_path)

src_folder = "/Workspace" + parent_path + "/deployment-staging/"
dst_folder = "/Workspace" + grandparent_path + "/deployment-digital-twin"

print(f"Source: {src_folder}")
print(f"Destination: {dst_folder}")

In [ ]:
# Fix the JavaScript file - replace BACKEND_URL_PLACEHOLDER
js_file_path = src_folder + "dist/static/js/main.90f88fa5.js"

print(f"Fixing JavaScript file: {js_file_path}")

# Read the file
with open(js_file_path, 'r') as f:
    content = f.read()

# Check if placeholder exists
if "BACKEND_URL_PLACEHOLDER" in content:
    print("⚠️  Found BACKEND_URL_PLACEHOLDER - replacing...")
    
    # Replace placeholder with actual app URL
    updated_content = content.replace("BACKEND_URL_PLACEHOLDER", app.url)
    
    # Write back
    with open(js_file_path, 'w') as f:
        f.write(updated_content)
    
    print(f"✅ Successfully replaced BACKEND_URL_PLACEHOLDER with {app.url}")
else:
    print("✅ BACKEND_URL_PLACEHOLDER already replaced (or not found)")

# Verify
with open(js_file_path, 'r') as f:
    content = f.read()
    if app.url in content:
        print(f"✅ Verified: App URL {app.url} is present in JavaScript file")
    else:
        print(f"❌ Warning: App URL not found in JavaScript file")

In [ ]:
# Copy the fixed deployment-staging to deployment folder
import shutil

if os.path.exists(dst_folder):
    shutil.rmtree(dst_folder)
    print(f"Removed existing deployment folder: {dst_folder}")

shutil.copytree(src_folder, dst_folder)
print(f"✅ Copied {src_folder} to {dst_folder}")

APP_SRC_CODE_PATH = os.path.abspath(dst_folder)
print(f"App source code path: {APP_SRC_CODE_PATH}")

In [ ]:
# Create new deployment
print("🔄 Creating new deployment...")

new_deployment = AppDeployment(
    deployment_id=str(uuid.uuid4()),
    mode=AppDeploymentMode.AUTO_SYNC,
    source_code_path=APP_SRC_CODE_PATH,
    deployment_artifacts=AppDeploymentArtifacts(source_code_path=APP_SRC_CODE_PATH)
)

try:
    w.apps.deploy(APP_NAME, new_deployment)
    print(f"✅ App {APP_NAME} has been redeployed successfully!")
    print(f"\n🌐 App URL: {app.url}")
    print("\n⏳ Wait 1-2 minutes for the deployment to complete.")
    print("Then refresh the app in your browser to see the fix.")
except Exception as e:
    print(f"❌ Failed to redeploy {APP_NAME}: {e}")

In [ ]:
# Display the app URL for easy access
displayHTML(f"""
<div style="padding: 20px; background-color: #e8f5e9; border-left: 4px solid #4caf50; margin: 10px 0;">
    <h3 style="color: #2e7d32; margin-top: 0;">✅ Deployment Complete</h3>
    <p style="font-size: 16px;">Your app has been redeployed with the backend URL fix.</p>
    <p style="font-size: 16px;"><strong>App URL:</strong> <a href="{app.url}" target="_blank">{app.url}</a></p>
    <p style="font-size: 14px; color: #666;">⏳ Allow 1-2 minutes for the deployment to propagate, then refresh your browser.</p>
</div>
""")